In [27]:
from torch_geometric import seed_everything

import pathpyG as pp

seed_everything(42)

In [28]:
custom_red = (213, 94, 0)
custom_pink = (204, 120, 188)

In [29]:
t_1 = pp.TemporalGraph.from_edge_list(
    [
        ("A", "B", 0),
        ("C", "D", 0),
        ("B", "C", 1),
        ("B", "A", 1),
        ("D", "C", 1),
        ("A", "C", 2),
        ("C", "B", 2),
        ("C", "B", 3),
    ],
)

edge_color = {
    ("A", "B", 0): custom_red,
    ("C", "D", 0): custom_red,
    ("C", "B", 2): custom_red,
}
node_opacity = {(node_id, time): 0.1 for node_id in t_1.nodes for time in range(t_1.data.time.max().item() + 2)}
node_opacity.update({(source_id, time): 0.75 for source_id, target_id, time in t_1.temporal_edges})
node_opacity.update({(target_id, time + 1): 0.75 for source_id, target_id, time in t_1.temporal_edges})
pp.plot(
    t_1,
    backend="tikz",
    kind="unfolded",
    edge_color=edge_color,
    edge_opacity=0.8,
    node_opacity=node_opacity,
    filename="t_1_unfolded.tex",
    width="2.5cm",
    height="1.5cm",
    margin=0.2,
    node_size=4,
    edge_size=1,
    orientation="right",
)

In [30]:
t_2 = pp.TemporalGraph.from_edge_list(
    [
        ("C", "B", 0),
        ("B", "C", 1),
        ("B", "A", 1),
        ("D", "C", 1),
        ("A", "B", 2),
        ("A", "C", 2),
        ("C", "D", 2),
        ("C", "B", 3)
    ],
)

edge_color = {
    ("A", "B", 2): custom_pink,
    ("C", "D", 2): custom_pink,
    ("C", "B", 0): custom_pink,
}
node_opacity = {(node_id, time): 0.1 for node_id in t_2.nodes for time in range(t_2.data.time.max().item() + 2)}
node_opacity.update({(source_id, time): 0.75 for source_id, target_id, time in t_2.temporal_edges})
node_opacity.update({(target_id, time + 1): 0.75 for source_id, target_id, time in t_2.temporal_edges})
pp.plot(
    t_2,
    backend="tikz",
    kind="unfolded",
    edge_color=edge_color,
    edge_opacity=0.8,
    node_opacity=node_opacity,
    filename="t_2_unfolded.tex",
    width="2.5cm",
    height="1.5cm",
    margin=0.2,
    node_size=4,
    edge_size=1,
    orientation="right",
)

In [31]:
eg_1 = pp.EventGraph.from_temporal_graph(t_1)
eg_2 = pp.EventGraph.from_temporal_graph(t_2)

# Event Graph layout similar to unfolded layout of temporal graph
layout_eg_1 = {
    node_id: (eg_1.event_time(i), eg_1.first_order_mapping.to_idx(node_id.split("->")[0])*eg_1.n_first_order + eg_1.first_order_mapping.to_idx(node_id.split("->")[1].split("@")[0]))
    for i, node_id in enumerate(eg_1.nodes)
}

layout_eg_2 = {
    node_id: (eg_2.event_time(i), eg_2.first_order_mapping.to_idx(node_id.split("->")[0])*eg_2.n_first_order + eg_2.first_order_mapping.to_idx(node_id.split("->")[1].split("@")[0]))
    for i, node_id in enumerate(eg_2.nodes)
}

100%|██████████| 4/4 [00:00<00:00, 891.27it/s]


In [32]:
node_color = {
    ("A->B@0"): custom_red,
    ("C->D@0"): custom_red,
    ("C->B@2"): custom_red,
}

edge_color = {
    ("A->B@0", "B->C@1"): custom_red,
    ("A->B@0", "B->A@1"): custom_red,
    ("C->D@0", "D->C@1"): custom_red,
    ("D->C@1", "C->B@2"): custom_red,
    ("B->C@1", "C->B@2"): custom_red,
}

pp.plot(
    eg_1,
    backend="tikz",
    layout=layout_eg_1,
    edge_size=2,
    edge_opacity=0.8,
    edge_color=edge_color,
    node_color=node_color,
    filename="event_1.tex",
    width = "4cm",
    height = "2cm",
    margin=0.2,
    node_size=8
)

In [33]:
node_color = {
    ("A->B@2"): custom_pink,
    ("C->D@2"): custom_pink,
    ("C->B@0"): custom_pink,
}
edge_color = {
    ("C->B@0", "B->C@1"): custom_pink,
    ("C->B@0", "B->A@1"): custom_pink,
    ("B->C@1", "C->D@2"): custom_pink,
    ("D->C@1", "C->D@2"): custom_pink,
    ("B->A@1", "A->B@2"): custom_pink
}

pp.plot(
    eg_2,
    backend="tikz",
    layout=layout_eg_2,
    edge_size=2,
    edge_opacity=0.8,
    edge_color=edge_color,
    node_color=node_color,
    filename="event_2.tex",
    width = "4cm",
    height = "2cm",
    margin=0.2,
    node_size=8
)

In [34]:
m_1 = pp.MultiOrderModel.from_temporal_graph(t_1, max_order=2, delta=1)
m_2 = pp.MultiOrderModel.from_temporal_graph(t_2, max_order=2, delta=1)

layout = pp.layout(m_1.layers[2] + m_2.layers[2], layout="fa2")

100%|██████████| 4/4 [00:00<00:00, 940.85it/s]


In [35]:
edge_color = {
    ("A->B", "B->C"): custom_red,
    ("A->B", "B->A"): custom_red,
    ("B->C", "C->B"): custom_red,
    ("C->D", "D->C"): custom_red,
    ("D->C", "C->B"): custom_red,
}

pp.plot(
    m_1.layers[2],
    backend="tikz",
    layout=layout,
    edge_size=2,
    edge_opacity=0.8,
    edge_color=edge_color,
    filename="ho_1.tex",
    width = "3cm",
    height = "2cm",
    margin=0.2,
    node_size=8
)

In [36]:
edge_color = {
    ("C->B", "B->A"): custom_pink,
    ("B->A", "A->B"): custom_pink,
    ("C->B", "B->C"): custom_pink,
    ("B->C", "C->D"): custom_pink,
    ("D->C", "C->D"): custom_pink,
}

pp.plot(
    m_2.layers[2],
    backend="tikz",
    layout=layout,
    edge_size=2,
    edge_opacity=0.8,
    edge_color=edge_color,
    filename="ho_2.tex",
    width = "3cm",
    height = "2cm",
    margin=0.2,
    node_size=8
)

In [37]:
static_1 = t_1.to_static_graph()
static_2 = t_2.to_static_graph()

layout_static = pp.layout(static_1 + static_2, layout="fa2")

In [38]:
# edge_color = {
#     ("A", "B"): custom_red,
#     ("C", "D"): custom_red,
#     ("C", "B"): custom_red,
# }

pp.plot(
    static_1,
    backend="tikz",
    layout=layout_static,
    edge_size=2,
    edge_opacity=0.8,
    # edge_color=edge_color,
    filename="static_1.tex",
    width="2cm",
    height="2cm",
    margin=0.2,
    node_size=7
)

In [39]:
# edge_color = {
#     ("C", "B"): custom_pink,
#     ("A", "B"): custom_pink,
#     ("C", "D"): custom_pink,
# }

# pp.plot(
#     static_2,
#     backend="tikz",
#     layout=layout_static,
#     edge_size=2,
#     edge_opacity=0.8,
#     edge_color=edge_color,
#     # filename="static_2.pdf",
#     width="4cm",
#     height="4cm",
#     margin=0.2,
#     node_size=12
# )